# Data Cleaning 01 -- Top 100 S&P 500 Universe (Annual)

## Input
`Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_annual.parquet` (2,100 rows x 4 columns: `permno`, `year`, `dlycap`, `rank`)

## Purpose
This is the first file cleaned in the pipeline. Everything downstream depends on it, so the focus is on thorough structural validation rather than imputation or transformation. Errors here would propagate to every other dataset.

## Stage 0: Load & Inspect
Basic shape, dtype, and date range verification. Confirms 2,100 rows (100 stocks x 21 years), 4 columns, year range 2004--2024.

## Stage 1: Missing Data Audit
- Total NaN count across all cells
- Per-column NaN counts
- Per-row NaN identification (any rows with missing values are printed)

## Stage 2: Structural Validation
Seven integrity checks are performed:

1. **Stocks per year** -- verifies exactly 100 unique PERMNOs per year
2. **No duplicate (permno, year) pairs** -- ensures each stock appears at most once per year
3. **Rank integrity** -- confirms ranks 1--100 with no gaps or extras in every year
4. **Market cap validity** -- checks for null, zero, or negative `dlycap` values
5. **Market cap ordering matches rank** -- verifies that `dlycap` is monotonically decreasing as rank increases within each year
6. **Year-over-year turnover** -- reports entries and exits per year, flags years with unusually high turnover (>15 entries)
7. **Market cap magnitude** -- sanity checks that values are in the expected range for top-100 S&P 500 stocks (CRSP `dlycap` is in thousands of dollars)

## Outcome
All structural checks passed with no issues found. The raw file is copied unchanged to the cleaned directory.

## Output
`Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet` (identical to input)

In [2]:
# %% [markdown]
# # Data Cleaning 01: Top-100 S&P 500 Universe
#
# Source: Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_annual.parquet
# Output: Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/
#           - universe_annual_clean.parquet
#           - universe_master_clean.parquet
#
# This is the first file to clean. Everything downstream depends on it.
# The universe file is small (2,100 rows) and structurally simple, but
# we need to verify its integrity thoroughly because errors here propagate
# to every other dataset.

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_annual.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns and dtypes:")
for c in df.columns:
    print(f"  {c:<15s} {str(df[c].dtype):<15s}")

print(f"\nYear range: {df['year'].min()} → {df['year'].max()}")
print(f"Unique years: {df['year'].nunique()}")
print(f"Unique PERMNOs: {df['permno'].nunique()}")

print(f"\n--- Head (5 rows) ---")
print(df.head(5).to_string(index=False))

print(f"\n--- Tail (5 rows) ---")
print(df.tail(5).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_nan = df.isna().sum().sum()
total_cells = df.shape[0] * df.shape[1]
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN ───────────────────────────────────────────────────────────
print(f"\n--- Per-Column NaN ---")
for c in df.columns:
    n = df[c].isna().sum()
    pct = n / len(df) * 100
    status = "✓" if n == 0 else "⚠"
    print(f"  {status} {c:<15s} {n:>5d} NaN ({pct:.1f}%)")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df.isna().sum(axis=1)
print(f"\n--- Per-Row NaN ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():,} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with any NaN: {(row_nan > 0).sum():,}")
if (row_nan > 0).any():
    print(f"\n  Rows with NaN:")
    print(df[row_nan > 0].to_string(index=False))


STAGE 0: LOAD & INSPECT

Shape: 2,100 rows × 4 columns

Columns and dtypes:
  permno          Int64          
  year            int64          
  dlycap          Float64        
  rank            int64          

Year range: 2004 → 2024
Unique years: 21
Unique PERMNOs: 227

--- Head (5 rows) ---
 permno  year        dlycap  rank
  12060  2004   311065842.8     1
  10107  2004   295294930.0     2
  11850  2004   271001800.0     3
  21936  2004  269621707.59     4
  70519  2004  250402181.58     5

--- Tail (5 rows) ---
 permno  year       dlycap  rank
  53613  2024  94207594.06    96
  45751  2024  93422351.84    97
  64390  2024  93185330.48    98
  92108  2024   93024551.4    99
  13511  2024   92975664.0   100

STAGE 1: MISSING DATA AUDIT

Total cells: 8,400
Total NaN:   0 (0.00%)

--- Per-Column NaN ---
  ✓ permno              0 NaN (0.0%)
  ✓ year                0 NaN (0.0%)
  ✓ dlycap              0 NaN (0.0%)
  ✓ rank                0 NaN (0.0%)

--- Per-Row NaN ---
  Rows with 0

In [3]:

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: STRUCTURAL VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════
#
# For the universe file, "cleaning" is mostly about validation rather than
# imputation. We need to verify structural properties that all downstream
# code assumes are true.

# %%
print("\n" + "=" * 90)
print("STAGE 2: STRUCTURAL VALIDATION")
print("=" * 90)

issues = []

# ── Check 1: Exactly 100 stocks per year ─────────────────────────────────────
stocks_per_year = df.groupby('year')['permno'].nunique()
print(f"\n--- Check 1: Stocks per year (expect 100) ---")
for year, n in stocks_per_year.items():
    status = "✓" if n == 100 else "⚠"
    if n != 100:
        issues.append(f"Year {year}: {n} stocks instead of 100")
    print(f"  {status} {year}: {n}")

# ── Check 2: No duplicate (permno, year) ────────────────────────────────────
n_dupes = df.duplicated(subset=['permno', 'year']).sum()
print(f"\n--- Check 2: Duplicate (permno, year) pairs ---")
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates found")
    issues.append(f"{n_dupes} duplicate (permno, year) pairs")
    print(df[df.duplicated(subset=['permno', 'year'], keep=False)]
          .sort_values(['permno', 'year']).to_string(index=False))

# ── Check 3: Ranks are 1-100 with no gaps ───────────────────────────────────
print(f"\n--- Check 3: Rank integrity ---")
for year in sorted(df['year'].unique()):
    year_ranks = sorted(df[df['year'] == year]['rank'].tolist())
    expected = list(range(1, 101))
    if year_ranks == expected:
        continue  # only print problems
    else:
        missing_ranks = set(expected) - set(year_ranks)
        extra_ranks = set(year_ranks) - set(expected)
        msg = f"Year {year}: "
        if missing_ranks:
            msg += f"missing ranks {sorted(missing_ranks)}"
        if extra_ranks:
            msg += f" extra ranks {sorted(extra_ranks)}"
        print(f"  ⚠ {msg}")
        issues.append(msg)

if not any("Check 3" in str(i) for i in issues):
    print(f"  ✓ All years have ranks 1-100 with no gaps")

# ── Check 4: Market cap is positive and non-null ────────────────────────────
print(f"\n--- Check 4: Market cap validity ---")
n_null_cap = df['dlycap'].isna().sum()
n_zero_cap = (df['dlycap'] <= 0).sum()
n_negative = (df['dlycap'] < 0).sum()
print(f"  Null dlycap:     {n_null_cap}")
print(f"  Zero dlycap:     {n_zero_cap}")
print(f"  Negative dlycap: {n_negative}")
if n_null_cap > 0 or n_zero_cap > 0:
    issues.append(f"{n_null_cap} null and {n_zero_cap} zero/negative market caps")
    print(f"\n  Rows with problematic dlycap:")
    print(df[df['dlycap'].isna() | (df['dlycap'] <= 0)].to_string(index=False))
else:
    print(f"  ✓ All market caps are positive")

# ── Check 5: Market cap ordering matches rank ───────────────────────────────
print(f"\n--- Check 5: Market cap ordering matches rank ---")
misranked_years = []
for year in sorted(df['year'].unique()):
    sub = df[df['year'] == year].sort_values('rank')
    # dlycap should be monotonically decreasing as rank increases
    caps = sub['dlycap'].values
    if not all(caps[i] >= caps[i+1] for i in range(len(caps)-1)):
        misranked_years.append(year)

if misranked_years:
    print(f"  ⚠ Years with rank/cap mismatch: {misranked_years}")
    issues.append(f"Rank/cap mismatch in years: {misranked_years}")
else:
    print(f"  ✓ Market cap decreases monotonically with rank in all years")

# ── Check 6: PERMNO stability (year-over-year turnover) ─────────────────────
print(f"\n--- Check 6: Year-over-year turnover ---")
prev_set = None
for year in sorted(df['year'].unique()):
    curr_set = set(df[df['year'] == year]['permno'])
    if prev_set is not None:
        entered = len(curr_set - prev_set)
        exited = len(prev_set - curr_set)
        flag = " ← HIGH" if entered > 15 else ""
        print(f"  {year}: +{entered:>2d} entered, -{exited:>2d} exited{flag}")
        if entered > 20:
            issues.append(f"Year {year}: unusually high turnover ({entered} entries)")
    prev_set = curr_set

# ── Check 7: dlycap units and magnitude ─────────────────────────────────────
print(f"\n--- Check 7: Market cap magnitude (sanity check) ---")
print(f"  Overall: min={df['dlycap'].min():.0f}, "
      f"median={df['dlycap'].median():.0f}, "
      f"max={df['dlycap'].max():.0f}")
# CRSP dlycap is in thousands of dollars
# Top-100 S&P 500 stocks should have market caps of ~$10B-$3T
# In thousands: ~10,000,000 to ~3,000,000,000
for sample_year in [2004, 2014, 2024]:
    sub = df[df['year'] == sample_year]
    rank1 = sub[sub['rank'] == 1]['dlycap'].iloc[0]
    rank100 = sub[sub['rank'] == 100]['dlycap'].iloc[0]
    print(f"  {sample_year}: Rank 1 = {rank1:,.0f}  Rank 100 = {rank100:,.0f}  "
          f"(in CRSP units, thousands $)")

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'=' * 50}")
if issues:
    print(f"ISSUES FOUND ({len(issues)}):")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
else:
    print("✓ ALL STRUCTURAL CHECKS PASSED — NO ISSUES FOUND")



STAGE 2: STRUCTURAL VALIDATION

--- Check 1: Stocks per year (expect 100) ---
  ✓ 2004: 100
  ✓ 2005: 100
  ✓ 2006: 100
  ✓ 2007: 100
  ✓ 2008: 100
  ✓ 2009: 100
  ✓ 2010: 100
  ✓ 2011: 100
  ✓ 2012: 100
  ✓ 2013: 100
  ✓ 2014: 100
  ✓ 2015: 100
  ✓ 2016: 100
  ✓ 2017: 100
  ✓ 2018: 100
  ✓ 2019: 100
  ✓ 2020: 100
  ✓ 2021: 100
  ✓ 2022: 100
  ✓ 2023: 100
  ✓ 2024: 100

--- Check 2: Duplicate (permno, year) pairs ---
  ✓ No duplicates

--- Check 3: Rank integrity ---
  ✓ All years have ranks 1-100 with no gaps

--- Check 4: Market cap validity ---
  Null dlycap:     0
  Zero dlycap:     0
  Negative dlycap: 0
  ✓ All market caps are positive

--- Check 5: Market cap ordering matches rank ---
  ✓ Market cap decreases monotonically with rank in all years

--- Check 6: Year-over-year turnover ---
  2005: + 9 entered, - 9 exited
  2006: +12 entered, -12 exited
  2007: + 9 entered, - 9 exited
  2008: +13 entered, -13 exited
  2009: +23 entered, -23 exited ← HIGH
  2010: +14 entered, -14 ex

In [5]:
import shutil
from pathlib import Path

src = Path('../../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_annual.parquet')
dst = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(src, dst)
print(f"Copied to {dst}")



Copied to ..\..\..\Data\Data_Collection\Cleaned\01_Top100_SP500_Universe\universe_annual_clean.parquet
